# corrector_02 - Frequency baseline and bigram + Viterbi (role B)

1. Count unigrams and bigrams on `data/corpus/train.txt`, save the model to `checkpoints/corrector/ngram.pkl.gz`.
2. Measure p_same for the emission model on noisy training pairs.
3. Compare **no correction**, **frequency table** and **Viterbi** on noisy validation sentences, in two settings:
   mixed noise (the default noise generator) and fully stripped text (no diacritics at all).

CPU runtime is enough. Needs the `GITHUB_TOKEN` secret with notebook access on.

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

token = userdata.get("GITHUB_TOKEN")
BRANCH = "b/corrector"  # change to "main" after the code is merged
!rm -rf /content/repo
!git clone -q -b {BRANCH} https://{token}@github.com/wickyhien18/Vietnamese-Handwriting-OCR.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

DRIVE_ROOT = "/content/drive/MyDrive/Vietnamese-Handwriting-OCR"
DATA_DIR = f"{DRIVE_ROOT}/data"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
OUT_DIR = f"{DRIVE_ROOT}/outputs"

In [ ]:
MAX_TRAIN = 1_000_000   # training sentences to count (lower it if RAM runs out)
MIN_BIGRAM = 2          # drop bigrams seen only once
K = 8                   # candidates per syllable in Viterbi
N_EVAL = 3000           # validation sentences to evaluate (Viterbi is pure Python, keep it moderate)
CORPUS_DIR = f"{DATA_DIR}/corpus"
MODEL_PATH = f"{CKPT_DIR}/corrector/ngram.pkl.gz"

## 1. Train the n-gram model

In [ ]:
import itertools, json, os, time
from src.corrector.ngram import NgramModel

start = time.time()
model = NgramModel(interpolation=0.8)
with open(f"{CORPUS_DIR}/train.txt", encoding="utf-8") as f:
    model.train(line.strip() for line in itertools.islice(f, MAX_TRAIN))
print(f"counted in {time.time() - start:.0f} s")
print("words:", model.total_words, " distinct syllables:", len(model.unigram),
      " plain keys:", len(model.forms), " bigrams:", len(model.bigram))

model.prune(MIN_BIGRAM)
print("bigrams after pruning:", len(model.bigram))

os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
model.save(MODEL_PATH)
print("saved", MODEL_PATH, f"{os.path.getsize(MODEL_PATH) / 1e6:.0f} MB")

## 2. Emission probability p_same, measured on noisy training pairs

In [ ]:
from src.corrector.noise import make_pairs, NoiseConfig
from src.corrector.viterbi import Emission, estimate_p_same

with open(f"{CORPUS_DIR}/train.txt", encoding="utf-8") as f:
    some_train = [line.strip() for line in itertools.islice(f, 20_000)]
p_same = estimate_p_same(make_pairs(some_train, seed=1))
print("p_same =", round(p_same, 4))
emission = Emission(p_same)

# Saved next to the model, so src.corrector.api.load(method="viterbi") can rebuild the same corrector.
with open(f"{CKPT_DIR}/corrector/viterbi.json", "w", encoding="utf-8") as f:
    json.dump({"p_same": p_same, "k": K}, f)

## 3. Evaluate on validation sentences

In [ ]:
import json
from src.corrector.correctors import IdentityCorrector, FrequencyCorrector, ViterbiCorrector
from src.corrector.evaluate import evaluate, print_table

with open(f"{CORPUS_DIR}/val.txt", encoding="utf-8") as f:
    val = [line.strip() for line in itertools.islice(f, N_EVAL)]

correctors = {
    "none": IdentityCorrector(),
    "frequency": FrequencyCorrector(model),
    "viterbi": ViterbiCorrector(model, emission, k=K),
}
settings = {
    "mixed noise": NoiseConfig(),
    "fully stripped": NoiseConfig(p_strip_sentence=1.0),
}

all_results = {}
for setting, config in settings.items():
    pairs = list(make_pairs(val, seed=123, config=config))  # fixed seed: same noise every run
    rows = {}
    for name, corrector in correctors.items():
        start = time.time()
        rows[name] = evaluate(corrector, pairs, model)
        rows[name]["seconds"] = round(time.time() - start, 1)
    print(f"\n=== {setting}  ({rows['none']['n_syllables']} syllables) ===")
    print_table(rows)
    all_results[setting] = rows

os.makedirs(OUT_DIR, exist_ok=True)
with open(f"{OUT_DIR}/corrector_ngram_eval.json", "w", encoding="utf-8") as f:
    json.dump({"p_same": p_same, "max_train": MAX_TRAIN, "k": K, "n_eval": N_EVAL,
               "results": all_results}, f, ensure_ascii=False, indent=2)
print("\nsaved", f"{OUT_DIR}/corrector_ngram_eval.json")

## 4. A few examples

In [ ]:
viterbi_corrector = correctors["viterbi"]
freq_corrector = correctors["frequency"]
for noisy, clean in list(make_pairs(val[:8], seed=5)):
    out = viterbi_corrector.correct(noisy)
    print("clean    :", clean)
    print("noisy    :", noisy)
    print("frequency:", freq_corrector.correct(noisy)["text"])
    print("viterbi  :", out["text"])
    print("conf     :", [round(c, 2) for c in out["conf"]])
    print()